In [1]:
from dataclasses import dataclass

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torchmetrics.classification import (
    BinaryAccuracy,
    BinaryF1Score,
    BinaryConfusionMatrix,
    BinaryMatthewsCorrCoef,
)
import plotly.express as px

In [2]:
device = "cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu"

# MLP

In [3]:
@dataclass
class MLPConfig:
    layers: list[nn.Module]

In [4]:
class MLP(nn.Module):
    def __init__(self, config: MLPConfig):
        super().__init__()
        self.model = nn.Sequential(*config.layers)

    def forward(self, xs):
        return self.model(xs)

In [5]:
def create_model(config: MLPConfig) -> MLP:
    return MLP(config).to(device)


# Training

In [6]:
@dataclass
class Hyperparameters:
    epochs: int
    lr: float
    batch_size: int
    patience: int

In [7]:
class EarlyStopping:
    def __init__(self, patience):
        self.patience = patience
        self.best_loss = float('inf')
        self.counter = 0
        self.should_stop = False

    def step(self, val_loss, model):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1

        if self.counter >= self.patience:
            self.should_stop = True

In [8]:
@dataclass
class Results:
    train_losses: list[float]
    test_losses: list[float]
    accuracies: list[float]
    f1s: list[float]
    sensitivities: list[float]
    specificities: list[float]
    mccs: list[float]

In [9]:
def train_test(model: MLP, H: Hyperparameters, X_train, y_train, X_test, y_test) -> Results:
    results = Results([], [], [], [], [], [], [])

    # Train mode
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=H.lr)
    early_stop = EarlyStopping(H.patience)
    train_ds = TensorDataset(X_train, y_train)
    train_loader = DataLoader(train_ds, batch_size=H.batch_size, shuffle=True)
    accuracy_fn = BinaryAccuracy().to(device)
    f1_fn = BinaryF1Score().to(device)
    cm_fn = BinaryConfusionMatrix().to(device)
    mcc_fn = BinaryMatthewsCorrCoef().to(device)

    for epoch in range(H.epochs):
        model.train()
        train_loss = 0

        for X_batch, y_batch in train_loader:
            logits = model(X_batch)
            loss = criterion(logits, y_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        results.train_losses.append(train_loss / len(train_loader))

        # Test mode
        model.eval()
        with torch.no_grad():
            logits = model(X_test)
            val_loss = criterion(logits, y_test)
            results.test_losses.append(val_loss.item())

            probs = F.sigmoid(logits)
            preds = (probs > 0.5).int()

            accuracy = accuracy_fn(preds, y_test)
            results.accuracies.append(accuracy.item())

            f1 = f1_fn(preds, y_test)
            results.f1s.append(f1.item())

            cm = cm_fn(preds, y_test)
            tn, fp, fn, tp = cm[0,0], cm[0,1], cm[1,0], cm[1,1]
            sensitivity = tp / (tp + fn)
            specificity = tn / (tn + fp)
            results.sensitivities.append(sensitivity.item())
            results.specificities.append(specificity.item())

            mcc = mcc_fn(preds, y_test)
            results.mccs.append(mcc.item())

        early_stop.step(val_loss, model)
        if early_stop.should_stop:
            print(f"\n⛔ Early stopping at epoch {epoch+1}")
            break

        print(f"Epoch {epoch + 1} out of {H.epochs}")

    del optimizer
    del criterion

    return results


In [10]:
def losses_plot(train_losses, test_losses):
    df = pd.DataFrame({
        "Epochs": list(range(len(train_losses))),
        "Train Loss": train_losses,
        "Test Loss": test_losses,
    })

    df_melted = df.melt(id_vars="Epochs", value_vars=["Train Loss", "Test Loss"],
                        var_name="Metric", value_name="Loss")

    fig = px.line(
        df_melted,
        x="Epochs",
        y="Loss",
        color="Metric",
        markers=True,
        title=f"Train & Test Loss vs Epochs"
    )
    fig.show()

In [11]:
def metrics_plot(accuracies, f1s, sensitivities, specificities, mccs):
    df = pd.DataFrame({
        "Epochs": list(range(len(accuracies))),
        "Accuracy": accuracies,
        "F1": f1s,
        "Sensitivity": sensitivities,
        "Specificity": specificities,
        "MCC": mccs,
    })

    df_melted = df.melt(id_vars="Epochs", value_vars=["Accuracy", "F1", "Sensitivity", "Specificity", "MCC"],
                        var_name="Metric", value_name="Score")

    fig = px.line(
        df_melted,
        x="Epochs",
        y="Score",
        color="Metric",
        markers=True,
        title="Val scores vs Epochs"
    )
    fig.show()

In [12]:
def confusion_matrix_plot(model: MLP, X_test, y_test):
    cm_fn = BinaryConfusionMatrix().to(device)
    logits = model(X_test)
    probs = F.sigmoid(logits)
    preds = (probs > 0.5).int()
    cm = cm_fn(y_test, preds)
    fig = px.imshow(cm.T.cpu(), text_auto=True)
    fig.show()

# Numerical

## Dataset

In [13]:
df = pd.read_pickle("../data/skin-lesion-data/numerical.pickle")

In [14]:
X = df.drop("malignant", axis=1).to_numpy()
y = df["malignant"].to_numpy()


In [15]:
X.shape

(773, 56)

In [16]:
y.shape

(773,)

In [17]:
y[:5]

array([1, 1, 1, 1, 0])

### Easier batching later

In [52]:
y = y[:, np.newaxis]
y.shape

(773, 1, 1)

In [53]:
def numpy_to_torch(x):
    return torch.from_numpy(x).float().to(device)

In [54]:
def train_val_test(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)
    X_test, X_val, y_test, y_val = train_test_split(X_test, y_test, test_size=0.5, random_state=42)
    X_train = numpy_to_torch(X_train)
    X_val = numpy_to_torch(X_val)
    X_test = numpy_to_torch(X_test)
    y_train = numpy_to_torch(y_train)
    y_val = numpy_to_torch(y_val)
    y_test = numpy_to_torch(y_test)
    return X_train, X_val, X_test, y_train, y_val, y_test

In [55]:
X_train, X_val, X_test, y_train, y_val, y_test = train_val_test(X, y)

In [56]:
X_train.shape, y_train.shape

(torch.Size([463, 120000]), torch.Size([463, 1, 1]))

In [57]:
X_val.shape, y_val.shape

(torch.Size([155, 120000]), torch.Size([155, 1, 1]))

In [58]:
X_test.shape, y_test.shape

(torch.Size([155, 120000]), torch.Size([155, 1, 1]))

In [59]:
model = create_model(
    MLPConfig(
        [
            nn.Linear(56, 100),
            nn.ReLU(),
            nn.Linear(100, 150),
            nn.Tanh(),
            nn.Linear(150, 1)
        ]
    )
)

## Overfitting

Making sure everything works by overfitting to the dataset

In [26]:
results = train_test(model, Hyperparameters(epochs=100, lr=1e-3, batch_size=24, patience=1000), X_train, y_train, X_val, y_val)

Epoch 1 out of 100
Epoch 2 out of 100
Epoch 3 out of 100
Epoch 4 out of 100
Epoch 5 out of 100
Epoch 6 out of 100
Epoch 7 out of 100
Epoch 8 out of 100
Epoch 9 out of 100
Epoch 10 out of 100
Epoch 11 out of 100
Epoch 12 out of 100
Epoch 13 out of 100
Epoch 14 out of 100
Epoch 15 out of 100
Epoch 16 out of 100
Epoch 17 out of 100
Epoch 18 out of 100
Epoch 19 out of 100
Epoch 20 out of 100
Epoch 21 out of 100
Epoch 22 out of 100
Epoch 23 out of 100
Epoch 24 out of 100
Epoch 25 out of 100
Epoch 26 out of 100
Epoch 27 out of 100
Epoch 28 out of 100
Epoch 29 out of 100
Epoch 30 out of 100
Epoch 31 out of 100
Epoch 32 out of 100
Epoch 33 out of 100
Epoch 34 out of 100
Epoch 35 out of 100
Epoch 36 out of 100
Epoch 37 out of 100
Epoch 38 out of 100
Epoch 39 out of 100
Epoch 40 out of 100
Epoch 41 out of 100
Epoch 42 out of 100
Epoch 43 out of 100
Epoch 44 out of 100
Epoch 45 out of 100
Epoch 46 out of 100
Epoch 47 out of 100
Epoch 48 out of 100
Epoch 49 out of 100
Epoch 50 out of 100
Epoch 51 

In [27]:
losses_plot(results.train_losses, results.test_losses)

In [28]:
metrics_plot(results.accuracies, results.f1s, results.sensitivities, results.specificities, results.mccs)

In [29]:
confusion_matrix_plot(model, X_val, y_val)

## Hyperparameter finetuning

In [32]:
configs = {
    "baseline": MLPConfig(
        [
            nn.Linear(56, 100),
            nn.ReLU(),
            nn.Linear(100, 150),
            nn.Tanh(),
            nn.Linear(150, 1),
        ]
    ),
    "shallow & narrow": MLPConfig(
        [
            nn.Linear(56, 150),
            nn.ReLU(),
            nn.Linear(150, 1),
        ]
    ),
    "deep & narrow": MLPConfig(
        [
            nn.Linear(56, 75),
            nn.ReLU(),
            nn.Linear(75, 100),
            nn.Tanh(),
            nn.Linear(100, 125),
            nn.ReLU(),
            nn.Linear(125, 150),
            nn.Tanh(),
            nn.Linear(150, 1),
        ]
    ),
    "shallow & wide": MLPConfig(
        [
            nn.Linear(56, 500),
            nn.ReLU(),
            nn.Linear(500, 1),
        ]
    ),
    "deep & wide": MLPConfig(
        [
            nn.Linear(56, 500),
            nn.ReLU(),
            nn.Linear(500, 1000),
            nn.Tanh(),
            nn.Linear(1000, 1500),
            nn.ReLU(),
            nn.Linear(1500, 2000),
            nn.Tanh(),
            nn.Linear(2000, 1),
        ]
    )
}

In [33]:
all_results = {}

for name, config in configs.items():
    print(f"Training model: {name}")

    model = create_model(config)
    results = train_test(
        model,
        Hyperparameters(epochs=100, lr=1e-3, batch_size=24, patience=10),
        X_train,
        y_train,
        X_val,
        y_val
    )

    all_results[name] = results

Training model: baseline
Epoch 1 out of 100
Epoch 2 out of 100
Epoch 3 out of 100
Epoch 4 out of 100
Epoch 5 out of 100
Epoch 6 out of 100
Epoch 7 out of 100
Epoch 8 out of 100
Epoch 9 out of 100
Epoch 10 out of 100
Epoch 11 out of 100
Epoch 12 out of 100
Epoch 13 out of 100
Epoch 14 out of 100
Epoch 15 out of 100
Epoch 16 out of 100
Epoch 17 out of 100
Epoch 18 out of 100

⛔ Early stopping at epoch 19
Training model: shallow & narrow
Epoch 1 out of 100
Epoch 2 out of 100
Epoch 3 out of 100
Epoch 4 out of 100
Epoch 5 out of 100
Epoch 6 out of 100
Epoch 7 out of 100
Epoch 8 out of 100
Epoch 9 out of 100
Epoch 10 out of 100
Epoch 11 out of 100
Epoch 12 out of 100
Epoch 13 out of 100
Epoch 14 out of 100
Epoch 15 out of 100
Epoch 16 out of 100
Epoch 17 out of 100
Epoch 18 out of 100
Epoch 19 out of 100
Epoch 20 out of 100
Epoch 21 out of 100
Epoch 22 out of 100
Epoch 23 out of 100
Epoch 24 out of 100
Epoch 25 out of 100
Epoch 26 out of 100
Epoch 27 out of 100
Epoch 28 out of 100
Epoch 29 o

In [34]:
summary = pd.DataFrame([
    {
        "model": name,
        "final_train_loss": res.train_losses[-1],
        "final_val_loss": res.test_losses[-1],
        "final_accuracy": res.accuracies[-1],
        "final_f1": res.f1s[-1],
        "final_sensitivity": res.sensitivities[-1],
        "final_specificity": res.specificities[-1],
        "final_mcc": res.mccs[-1],
    }
    for name, res in all_results.items()
])
summary = summary.set_index("model")

In [35]:
summary

,final_train_loss,final_val_loss,final_accuracy,final_f1,final_sensitivity,final_specificity,final_mcc
model,,,,,,,
baseline,0.105739,0.437192,0.858065,0.888889,0.926316,0.750000,0.697697
shallow & narrow,0.178823,0.389830,0.864516,0.887701,0.873684,0.850000,0.717666
deep & narrow,0.063463,0.771640,0.793548,0.820225,0.768421,0.833333,0.587694
shallow & wide,0.169468,0.384872,0.845161,0.869565,0.842105,0.850000,0.681777
deep & wide,0.071011,0.951359,0.832258,0.855556,0.810526,0.866667,0.662814


In [36]:
fig = px.imshow(summary[["final_train_loss", "final_val_loss"]], text_auto=True)
fig.show()

In [37]:
fig = px.imshow(summary.drop(columns=["final_train_loss", "final_val_loss"]), text_auto=True)
fig.show()

### Shallow & Narrow Wins!

In [38]:
model = create_model(
    MLPConfig(
        [
            nn.Linear(56, 150),
            nn.ReLU(),
            nn.Linear(150, 1),
        ]
    )
)

In [39]:
results = train_test(
    model,
    Hyperparameters(epochs=100, lr=1e-3, batch_size=24, patience=10),
    X_train,
    y_train,
    X_test,
    y_test
)

Epoch 1 out of 100
Epoch 2 out of 100
Epoch 3 out of 100
Epoch 4 out of 100
Epoch 5 out of 100
Epoch 6 out of 100
Epoch 7 out of 100
Epoch 8 out of 100
Epoch 9 out of 100
Epoch 10 out of 100
Epoch 11 out of 100
Epoch 12 out of 100
Epoch 13 out of 100
Epoch 14 out of 100
Epoch 15 out of 100
Epoch 16 out of 100
Epoch 17 out of 100
Epoch 18 out of 100
Epoch 19 out of 100
Epoch 20 out of 100
Epoch 21 out of 100
Epoch 22 out of 100
Epoch 23 out of 100
Epoch 24 out of 100
Epoch 25 out of 100
Epoch 26 out of 100
Epoch 27 out of 100
Epoch 28 out of 100
Epoch 29 out of 100
Epoch 30 out of 100
Epoch 31 out of 100
Epoch 32 out of 100
Epoch 33 out of 100
Epoch 34 out of 100

⛔ Early stopping at epoch 35


In [40]:
losses_plot(results.train_losses, results.test_losses)

In [41]:
metrics_plot(results.accuracies, results.f1s, results.sensitivities, results.specificities, results.mccs)

In [42]:
confusion_matrix_plot(model, X_test, y_test)

# Images

## Dataset

In [62]:
import os

from PIL import Image
from torchvision import transforms

In [63]:
df = pd.read_pickle("../data/skin-lesion-data/binary.pickle")

In [64]:
def load_images(df, root_dir):
    X = []
    y = []

    for _, row in df.iterrows():
        path = os.path.join(root_dir, row["img_id"])
        img = Image.open(path).convert("RGB")
        img = np.array(img) / 255.0
        mean = np.array([0.485, 0.456, 0.406])
        std  = np.array([0.229, 0.224, 0.225])
        # Normalize each channel
        img = (img - mean) / std
        flat = img.reshape(-1) # flatten to [3*H*W]
        X.append(flat)
        y.append(int(row["malignant"]))

    return np.array(X), np.array(y)

In [65]:
X, y = load_images(df, "../data/skin-lesion-data/images/train")

In [66]:
X.shape, y.shape

((773, 120000), (773,))

In [67]:
y[:5]

array([1, 1, 1, 1, 0])

### Easier batching later

In [68]:
y = y[:, np.newaxis]
y.shape

(773, 1)

In [69]:
img_size = X.shape[1]
img_size

120000

In [70]:
X_train, X_val, X_test, y_train, y_val, y_test = train_val_test(X, y)

In [71]:
X_train.shape, y_train.shape

(torch.Size([463, 120000]), torch.Size([463, 1]))

In [72]:
X_val.shape, y_val.shape

(torch.Size([155, 120000]), torch.Size([155, 1]))

In [73]:
X_test.shape, y_test.shape

(torch.Size([155, 120000]), torch.Size([155, 1]))

In [42]:
model = create_model(
    MLPConfig(
        [
            nn.Linear(img_size, 1024),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(512, 1)
        ]
    )
)

## Overfitting

Make sure the model works by overfitting

In [56]:
results = train_test(model, Hyperparameters(epochs=100, lr=1e-3, batch_size=24, patience=1000), X_train, y_train, X_val, y_val)

Epoch 1 out of 100
Epoch 2 out of 100
Epoch 3 out of 100
Epoch 4 out of 100
Epoch 5 out of 100
Epoch 6 out of 100
Epoch 7 out of 100
Epoch 8 out of 100
Epoch 9 out of 100
Epoch 10 out of 100
Epoch 11 out of 100
Epoch 12 out of 100
Epoch 13 out of 100
Epoch 14 out of 100
Epoch 15 out of 100
Epoch 16 out of 100
Epoch 17 out of 100
Epoch 18 out of 100
Epoch 19 out of 100
Epoch 20 out of 100
Epoch 21 out of 100
Epoch 22 out of 100
Epoch 23 out of 100
Epoch 24 out of 100
Epoch 25 out of 100
Epoch 26 out of 100
Epoch 27 out of 100
Epoch 28 out of 100
Epoch 29 out of 100
Epoch 30 out of 100
Epoch 31 out of 100
Epoch 32 out of 100
Epoch 33 out of 100
Epoch 34 out of 100
Epoch 35 out of 100
Epoch 36 out of 100
Epoch 37 out of 100
Epoch 38 out of 100
Epoch 39 out of 100
Epoch 40 out of 100
Epoch 41 out of 100
Epoch 42 out of 100
Epoch 43 out of 100
Epoch 44 out of 100
Epoch 45 out of 100
Epoch 46 out of 100
Epoch 47 out of 100
Epoch 48 out of 100
Epoch 49 out of 100
Epoch 50 out of 100
Epoch 51 

In [57]:
losses_plot(results.train_losses, results.test_losses)

In [58]:
metrics_plot(results.accuracies, results.f1s, results.sensitivities, results.specificities, results.mccs)

In [59]:
confusion_matrix_plot(model, X_val, y_val)

## Hyperparameter finetuning

In [ ]:
configs = {
    "baseline": MLPConfig(
        [
            nn.Linear(img_size, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.Tanh(),
            nn.Linear(512, 1),
        ]
    ),
    "shallow & narrow": MLPConfig(
        [
            nn.Linear(img_size, 512),
            nn.ReLU(),
            nn.Linear(512, 1),
        ]
    ),
    "deep & narrow": MLPConfig(
        [
            nn.Linear(img_size, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.Tanh(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.Tanh(),
            nn.Linear(512, 1),
        ]
    ),
    "shallow & wide": MLPConfig(
        [
            nn.Linear(img_size, 1024),
            nn.ReLU(),
            nn.Linear(1024, 1),
        ]
    ),
    "deep & wide": MLPConfig(
        [
            nn.Linear(img_size, 512),
            nn.ReLU(),
            nn.Linear(512, 750),
            nn.Tanh(),
            nn.Linear(750, 1024),
            nn.ReLU(),
            nn.Linear(1024, 750),
            nn.Tanh(),
            nn.Linear(750, 1),
        ]
    ),
}


In [ ]:

for name, config in configs.items():
    print(f"Training model: {name}")

    model = create_model(config)
    results = train_test(
        model,
        Hyperparameters(epochs=100, lr=1e-3, batch_size=24, patience=10),
        X_train,
        y_train,
        X_val,
        y_val
    )

    all_results[name] = results

Training model: shallow & wide
Epoch 1 out of 100
Epoch 2 out of 100
Epoch 3 out of 100
Epoch 4 out of 100
Epoch 5 out of 100
Epoch 6 out of 100
Epoch 7 out of 100
Epoch 8 out of 100
Epoch 9 out of 100
Epoch 10 out of 100
Epoch 11 out of 100
Epoch 12 out of 100
Epoch 13 out of 100
Epoch 14 out of 100
Epoch 15 out of 100
Epoch 16 out of 100
Epoch 17 out of 100
Epoch 18 out of 100

⛔ Early stopping at epoch 19
Training model: deep & wide
Epoch 1 out of 100
Epoch 2 out of 100
Epoch 3 out of 100
Epoch 4 out of 100
Epoch 5 out of 100
Epoch 6 out of 100
Epoch 7 out of 100
Epoch 8 out of 100
Epoch 9 out of 100
Epoch 10 out of 100
Epoch 11 out of 100
Epoch 12 out of 100

⛔ Early stopping at epoch 13


In [48]:
summary = pd.DataFrame([
    {
        "model": name,
        "final_train_loss": res.train_losses[-1],
        "final_val_loss": res.test_losses[-1],
        "final_accuracy": res.accuracies[-1],
        "final_f1": res.f1s[-1],
        "final_sensitivity": res.sensitivities[-1],
        "final_specificity": res.specificities[-1],
        "final_mcc": res.mccs[-1],
    }
    for name, res in all_results.items()
])
summary = summary.set_index("model")

In [49]:
summary

,final_train_loss,final_val_loss,final_accuracy,final_f1,final_sensitivity,final_specificity,final_mcc
model,,,,,,,
baseline,0.660246,0.667064,0.606452,0.755020,0.989474,0.000000,-0.064040
shallow & narrow,0.257438,3.725882,0.625806,0.733945,0.842105,0.283333,0.150953
shallow & wide,0.150413,3.838462,0.651613,0.763158,0.915789,0.233333,0.208135
deep & wide,0.650058,0.666203,0.619355,0.751055,0.936842,0.116667,0.094026


In [50]:
fig = px.imshow(summary[["final_train_loss", "final_val_loss"]], text_auto=True)
fig.show()

In [51]:
fig = px.imshow(summary.drop(columns=["final_train_loss", "final_val_loss"]), text_auto=True)
fig.show()

### Shallow & Wide wins!

In [74]:
model = create_model(
    MLPConfig(
        [
            nn.Linear(img_size, 1024),
            nn.ReLU(),
            nn.Linear(1024, 1),
        ]
    )
)

In [75]:
results = train_test(model, Hyperparameters(epochs=100, lr=1e-3, batch_size=24, patience=10), X_train, y_train, X_test, y_test)

Epoch 1 out of 100
Epoch 2 out of 100
Epoch 3 out of 100
Epoch 4 out of 100
Epoch 5 out of 100
Epoch 6 out of 100
Epoch 7 out of 100
Epoch 8 out of 100
Epoch 9 out of 100
Epoch 10 out of 100
Epoch 11 out of 100
Epoch 12 out of 100
Epoch 13 out of 100
Epoch 14 out of 100
Epoch 15 out of 100

⛔ Early stopping at epoch 16


In [76]:
losses_plot(results.train_losses, results.test_losses)

In [77]:
metrics_plot(results.accuracies, results.f1s, results.sensitivities, results.specificities, results.mccs)

In [78]:
confusion_matrix_plot(model, X_test, y_test)